# Desicion Tree Implementation - Titanic

In [ ]:
# Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Cargar datos

Mi matriz "X" serán las características necesarias para determinar superviviencia 

Mi vector de chequeo "y" será la columna: Survived

In [ ]:
PATH = "../data/titanic-machine-learning-from-disaster"

df = pd.read_csv(PATH + "/train.csv")

df.head(5)

### Normalizar Datos

In [ ]:
df.columns = df.columns.str.lower()

# Buscamos columnas con contenido en texto (type = string)
strings = list(df.select_dtypes(include=['string']).columns)

for col in strings:
    df[col] = df[col].str.lower().str.replace(' ', '_')

df.head()

## Análisis Exploratorio de Datos

### EDA: Sexo

In [ ]:
# Porcentaje de Hombres y Mujeres
supervivientes = df[df['survived'] == 1]
print(f"Total de Supervivientes: {len(supervivientes)}, Total en el barco: {len(df)}")

supervivientes_mujeres = df[(df['survived'] == 1) & (df['sex'] == 'female')]
print("\n Supervivientes Mujeres: ", len(supervivientes_mujeres))

supervivientes_hombres = len(supervivientes) - len(supervivientes_mujeres)
print("\n Supervivientes Hombres: ", supervivientes_hombres)

porcentaje_sexo = supervivientes['sex'].value_counts(normalize=True) * 100
print("\nPorcentaje por sexo entre supervivientes:\n", porcentaje_sexo)

total_por_sexo = df['sex'].value_counts()
print("\nCantidad total de cada sexo en el barco:\n", total_por_sexo)

### EDA: Edad

In [ ]:
# Porcentaje por Rango de Edad entre Sobrevivientes

# Crear copia sin nulos
df_edad = df.dropna(subset=['age']).copy()

# Rangos de edad y las etiquetas
rangos = [0, 12, 19, 59, 100]
etiquetas = ['Niños', 'Adolescentes', 'Adultos', 'Ancianos']

df_edad['rango_edad'] = pd.cut(df_edad['age'], bins = rangos, labels=etiquetas)

# Filtrar por supervivientes
supervivientes_edad = df_edad[df_edad['survived'] == 1]

# Porcentaje
porcentaje_edad = supervivientes_edad['rango_edad'].value_counts(normalize=True) * 100
print("Porcentaje por rango de edad entre supervivientes:\n", porcentaje_edad)

### EDA: Parch y SibSp

In [ ]:
# Importancia de Parch (Padres / Hijos)
print("--- Tasa de supervivencia según Parch (Padres/Hijos) ---")
parch_stats = df.groupby('parch')['survived'].agg(['count', 'mean'])
parch_stats['mean'] = parch_stats['mean'] * 100
print(parch_stats)

# Importancia de SibSp (Hermanos / Cónyugue)
print("\n--- Tasa de supervivencia según SibSp (Hermanos/Cónyuge) ---")
sibsp_stats = df.groupby('sibsp')['survived'].agg(['count', 'mean'])
sibsp_stats['mean'] = sibsp_stats['mean'] * 100 # Corrección menor de variable
print(sibsp_stats)

Las columnas que más información aportan a la hora de decidir quién sobrevive y quién no es sin lugar a dudas (en mi opinión): edad y sexo.

Entonces vamos a hacer un análisis más profundo de estas dos columnas.

### EDA: Cruzado Sexo/Edad

In [ ]:
# Análisis cruzado: Sexo y Rango de EDAD

analisis_cruzado = df_edad.groupby(['sex', 'rango_edad'])['survived'].agg(
    Total = 'count',
    Sobrevivieron = 'sum').reset_index()

# Los que Fallecieron
analisis_cruzado['Fallecieron'] = analisis_cruzado['Total'] - analisis_cruzado['Sobrevivieron']

# Porcentaje de supervivencia de ese grupo ESPECÍFICO
analisis_cruzado['Tasa_Supervivencia_%'] = (analisis_cruzado['Sobrevivieron'] / analisis_cruzado['Total']) * 100

# Crear el gráfico
plt.figure(figsize=(10, 6))
sns.barplot(data=df_edad, x='rango_edad', y='survived', hue='sex', errorbar=None)

# Personalización
plt.title('Tasa de Supervivencia por Edad y Sexo en el Titanic')
plt.ylabel('Tasa de Supervivencia (0 a 1)')
plt.xlabel('Rango de Edad')
plt.legend(title='Sexo')
plt.show()

# Modelo Árbol de Decisión

Primero vamos a extraer nuestra matriz **X** que tendrá las columnas **sex** y **age**.

Y también nuestro vector con el cual vamos a comparar y entrenar nuestro modelo, este vector es **y** y contendrá los resultados de la columna **survivde**.

In [ ]:
df_modelo = df.copy()  # Copia de nuestro DataFrame original

# Rellenar los valores nulos de 'age' con la mediana de las edades
# (Mejor que dropna, ya que no perdemos pasajeros)
mediana_edad = df_modelo['age'].median()
df_modelo['age'] = df_modelo['age'].fillna(mediana_edad)

# Vamos a codificar la columna sex (female: 1, male: 0)
df_modelo['sex'] = df_modelo['sex'].map({'female': 1, 'male': 0})

# Construimos 'X' con las columnas 'sex' y 'age'
X = df_modelo[['sex', 'age']]

# Construir nuestro target
y = df_modelo['survived']

### Implementación Árbol de Decisión

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn import tree

# Dividimos los datos: 80% para entrenar, 20% para probar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear modelo g(X)
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)

# Entrenar el modelo
arbol.fit(X_train, y_train)

# Evaluar el modelo
precission = arbol.score(X_test, y_test)
print(f"Precisión del modelo en datos de prueba: {precission * 100:.2f}%\n")

# Dibujar el árbol para ver su lógica
plt.figure(figsize=(12, 8))
tree.plot_tree(arbol, feature_names=['sex', 'age'], class_names=['Falleció', 'Sobrevivió'], filled=True, rounded=True, fontsize=10)
plt.title("Árbol de Decisión: Titanic (Sexo y Edad)")
plt.show()